# 00 Metadaten – Überblick über alle Versuche

Dieses Notebook liest **nur die Metadaten** jeder Messung (`meta/device.csv` und `meta/time.csv`), keine Sensordaten. Die Sensordaten werden in den Versuchs-Notebooks `V01.ipynb` … `V10.ipynb` analysiert.

In [1]:
import re
from pathlib import Path
import pandas as pd

# Path to the raw data – the only line you need to change on another computer
DATA_ROOT = Path('/Users/majasmereczynska/Documents/Industrial AI/IPR_Daten_neu/data/Hybridachse')
pd.set_option('display.width', 250, 'display.max_columns', 20, 'display.max_rows', 60, 'display.max_colwidth', 70)

In [2]:
# One row per recording folder (= every folder that has a meta/ subfolder).
# Only the two metadata files are read: device.csv (which phone) and time.csv (when it was recorded).
rows = []
for time_file in sorted(DATA_ROOT.glob('*/**/meta/time.csv')):
    folder = time_file.parent.parent
    device = pd.read_csv(folder / 'meta/device.csv').set_index('property')['value']
    events = pd.read_csv(time_file)
    rows.append({
        'Versuch': re.match(r'Versuch\d+', folder.relative_to(DATA_ROOT).parts[0]).group(),  # Versuch03_iPhone -> Versuch03
        'Ordner': str(folder.relative_to(DATA_ROOT)),
        'Gerät': 'Fairphone' if device['deviceModel'] == 'FP4' else 'iPhone',
        'Modell': device['deviceModel'],
        'OS': device['deviceRelease'],
        'App': device['version'],
        'Start': events['system time text'].iloc[0][11:19],                 # real clock time of START
        'Dauer (s)': round(events['experiment time'].iloc[-1], 1),
        'Pausen': (events['event'] == 'PAUSE').sum() - 1,                    # a normal recording ends with 1 PAUSE
        '_start_exact': events['system time'].iloc[0],                       # for the duplicate check below
    })
meta = pd.DataFrame(rows)

# Does the folder name match the device in the metadata? ('Fairphone'/'FP_' vs. 'iPhone'/'IP_')
says_iphone = meta['Ordner'].str.contains('iPhone|IP_')
says_fairphone = meta['Ordner'].str.contains('Fairphone|FP_|Schiene')
meta['Hinweis'] = ''
meta.loc[says_fairphone & (meta['Gerät'] == 'iPhone'), 'Hinweis'] = 'Ordnername sagt Fairphone, Daten sind iPhone'

# Duplicate = same phone and same start time to the millisecond -> the same recording saved twice.
# Correctly named folders are checked first, so the mislabelled copy is the one marked as duplicate.
meta['Duplikat von'] = ''
first = {}
for i, r in meta.sort_values(by='Hinweis', kind='stable').iterrows():
    key = (r['Modell'], r['_start_exact'])
    if key in first:
        meta.loc[i, 'Duplikat von'] = meta.loc[first[key], 'Ordner']
    else:
        first[key] = i
meta = meta.drop(columns='_start_exact')
meta

,Versuch,Ordner,Gerät,Modell,OS,App,Start,Dauer (s),Pausen,Hinweis,Duplikat von
0,Versuch01,Versuch01/Schiene,Fairphone,FP4,13,1.1.16,10:18:06,43.0,1,,
1,Versuch02,Versuch02/Alle Sensoren iPhone Achsenpos1 2024-12-09 11-15-19,iPhone,"iPhone13,2",18.1.1,1.1.16,11:12:47,54.9,0,,
2,Versuch02,Versuch02/Fairphone,Fairphone,FP4,13,1.1.16,11:12:51,50.1,0,,
3,Versuch03,Versuch03_Fairphone/Fairphone01,Fairphone,FP4,13,1.1.16,11:47:37,50.8,0,,
4,Versuch03,Versuch03_Fairphone/Fairphone02,Fairphone,FP4,13,1.1.16,11:51:33,52.6,0,,
5,Versuch03,Versuch03_Fairphone/Fairphone03,Fairphone,FP4,13,1.1.16,11:54:07,47.1,1,,
6,Versuch03,Versuch03_iPhone/Alle Sensoren 2024-12-09 11-50-01,iPhone,"iPhone13,2",18.1.1,1.1.16,11:47:33,54.9,0,,
7,Versuch03,Versuch03_iPhone/Alle Sensoren 2024-12-09 11-50-24,iPhone,"iPhone13,2",18.1.1,1.1.16,11:47:33,54.9,0,,Versuch03_iPhone/Alle Sensoren 2024-12-09 11-50-01
8,Versuch03,Versuch03_iPhone/Alle Sensoren 2024-12-09 11-52-34,iPhone,"iPhone13,2",18.1.1,1.1.16,11:51:38,49.0,0,,
9,Versuch03,Versuch03_iPhone/Alle Sensoren 2024-12-09 11-56-01,iPhone,"iPhone13,2",18.1.1,1.1.16,11:54:03,48.5,0,,


In [3]:
# Number of recordings per Versuch and device (duplicates not counted)
unique = meta[meta['Duplikat von'] == '']
pd.crosstab(unique['Versuch'], unique['Gerät'], margins=True, margins_name='Summe')

Gerät,Fairphone,iPhone,Summe
Versuch,,,
Versuch01,1,0,1
Versuch02,1,1,2
Versuch03,3,3,6
Versuch04,2,3,5
Versuch05,3,3,6
Versuch06,3,0,3
Versuch07,1,0,1
Versuch08,1,1,2
Versuch09,3,0,3


In [4]:
# Same phone every time? If model, OS and app version never change within a device, one physical
# phone per model was most likely used (phyphox stores no serial number, so this is not a proof).
meta.groupby('Gerät')[['Modell', 'OS', 'App']].nunique().rename(columns=lambda c: f'verschiedene {c}')

,verschiedene Modell,verschiedene OS,verschiedene App
Gerät,,,
Fairphone,1,1,1
iPhone,1,1,1


## Befund

- **38 Messordner** in 10 Versuchen. **5 davon sind Duplikate** (gleiches Handy, gleiche Startzeit auf die Millisekunde), es bleiben **33 Messungen**: 21 Fairphone, 12 iPhone.
  - Versuch03: 4 doppelt exportierte iPhone-Ordner.
  - Versuch04: `Versuch04_Fairphone_03` ist laut Metadaten ein **iPhone** und ein Duplikat von `Versuch04_iPhone_03`. Eine dritte Fairphone-Messung fehlt.
- **Versuch03** liegt in zwei Ordnern (`Versuch03_Fairphone`, `Versuch03_iPhone`), gehört aber zusammen.
- **Geräte:** immer dasselbe Fairphone 4 (Android 13) und dasselbe iPhone 12 (`iPhone13,2`, iOS 18.1.1), beide mit phyphox 1.1.16. Modell, OS und App-Version ändern sich nie.
- **Zeit:** alle Messungen am 09.12.2024 zwischen 10:18 und 16:16 Uhr, Dauer 40–87 s.
- **Gleichzeitige Messungen:** In mehreren Versuchen liefen Fairphone und iPhone gleichzeitig (Startzeiten wenige Sekunden auseinander). Das erlaubt einen direkten Vergleich der Geräte.
- **Pausen:** 3 Fairphone-Messungen wurden während der Aufnahme kurz pausiert (V01 Schiene, V03 Fairphone03, V05 FP_03).